# Bio-Semantic Drift Detector — Local Test
Runs the full CGD pipeline on 5 LLM-generated chains + 5 prewritten test cases.

| Stage | Cell | What it does |
|-------|------|--------------|
| 0 | 0–3 | Environment, API keys, imports, integrity checks |
| 1 | 4–5 | LLM CoT generation (5 chains) + 5 prewritten cases |
| 2 | 6 | Concept extraction (UMLS) |
| 3 | 7 | NLI entailment |
| 5 | 8 | UMLS density + specificity signals |
| 6 | 9–12 | Results table + visualisations |

**Kernel:** Bio-Semantic-Drift (venv)

In [ ]:
# ── 0. Environment setup ──────────────────────────────────────────────────────
import os, sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

try:
    from dotenv import load_dotenv
    load_dotenv(Path(PROJECT_ROOT) / '.env')
except ImportError:
    pass

# Auto-detect local UMLS DB
_local_db = str(Path(PROJECT_ROOT) / 'utils' / 'umls_local.db')
if Path(_local_db).exists():
    os.environ['UMLS_LOCAL_DB_PATH'] = _local_db
    print(f'  Local UMLS DB found  ->  {_local_db}')
    print('  UMLS signals will use local SQLite (no UMLS_API_KEY needed).')
else:
    print('  WARNING: utils/umls_local.db not found.')
    print('  UMLS calls will use REST API — set UMLS_API_KEY below.')

os.environ.pop('FORCE_HEURISTIC_NLI', None)
print('Environment ready.')

In [ ]:
# ── 1. API Key Input ──────────────────────────────────────────────────────────
# OpenRouter key  — required for LLM CoT generation (5 chains).
# UMLS API key    — only needed if utils/umls_local.db is absent.
# Keys already in .env are pre-loaded; only fill what is missing.
import ipywidgets as widgets
from IPython.display import display, HTML

_needs_umls = not os.getenv('UMLS_LOCAL_DB_PATH')
_layout     = widgets.Layout(width='480px')

_or_box  = widgets.Password(
    placeholder='sk-or-v1-...  (get a free key at openrouter.ai)',
    description='OpenRouter:', layout=_layout)
_btn     = widgets.Button(description='Apply Keys', button_style='primary',
                           icon='check')
_status  = widgets.Output()

def _apply(b):
    with _status:
        _status.clear_output()
        if _or_box.value.strip():
            os.environ['OPENROUTER_API_KEY'] = _or_box.value.strip()
            print(f'  OpenRouter key set ({len(_or_box.value.strip())} chars).')
        if _needs_umls and _umls_box.value.strip():
            os.environ['UMLS_API_KEY'] = _umls_box.value.strip()
            print(f'  UMLS API key set.')
        print('Done — re-run the imports cell (cell 2) to patch the cot_generator module.')

_btn.on_click(_apply)
_widget_list = [_or_box]
if _needs_umls:
    _umls_box = widgets.Password(
        placeholder='UMLS API key  (uts.nlm.nih.gov)',
        description='UMLS:', layout=_layout)
    _widget_list.append(_umls_box)
_widget_list += [_btn, _status]
display(widgets.VBox(_widget_list))

In [ ]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import warnings, time, re as _re, math, importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Reload cot_generator so it picks up OPENROUTER_API_KEY set in cell 1
import utils.cot_generator as _cg_mod
importlib.reload(_cg_mod)
_cg_mod.OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
_cg_mod.OPENROUTER_READY   = bool(_cg_mod.OPENROUTER_API_KEY)
_cg_mod.ANTHROPIC_API_KEY  = os.environ.get('ANTHROPIC_API_KEY', '')
_cg_mod.ANTHROPIC_READY    = bool(_cg_mod.ANTHROPIC_API_KEY)
from utils.cot_generator import generate as generate_cot

from utils.concept_extractor_v2    import extract_concepts
from utils.hybrid_checker          import build_entailment_records, _ensure_model
import utils.hybrid_checker        as _hc
from utils.umls_density_scorer     import score_density
from utils.umls_specificity_scorer import score_specificity
from utils                         import umls_api_linker

print('All modules imported.')
print(f'  OpenRouter ready : {_cg_mod.OPENROUTER_READY}')
print(f'  Anthropic ready  : {_cg_mod.ANTHROPIC_READY}')
print(f'  UMLS configured  : {umls_api_linker.is_configured()}')

In [ ]:
# ── 3. Integrity checks ───────────────────────────────────────────────────────
checks = []
PASS, FAIL, WARN = 'PASS', 'FAIL', 'WARN'

def _chk(label, ok, note=''):
    checks.append((PASS if ok else FAIL, label, note))
def _warn(label, note=''):
    checks.append((WARN, label, note))

# Module imports
for _mod in ['utils.cot_generator', 'utils.concept_extractor_v2',
             'utils.hybrid_checker', 'utils.umls_density_scorer',
             'utils.umls_specificity_scorer']:
    try:
        __import__(_mod)
        _chk(f'{_mod} importable', True)
    except Exception as e:
        _chk(f'{_mod} importable', False, str(e))

# ML deps
for _pkg in ['torch', 'transformers', 'peft']:
    try:
        __import__(_pkg)
        _chk(f'{_pkg} installed', True)
    except ImportError:
        _chk(f'{_pkg} installed', False, f'pip install {_pkg}')

# API keys
_chk('OpenRouter key set',
     bool(os.getenv('OPENROUTER_API_KEY') or os.getenv('ANTHROPIC_API_KEY')),
     'needed for LLM generation — paste in cell 1')
_chk('UMLS configured', umls_api_linker.is_configured(),
     'set UMLS_API_KEY or build utils/umls_local.db')
_local_db_path = os.getenv('UMLS_LOCAL_DB_PATH', '')
_chk('Local UMLS DB active',
     bool(_local_db_path) and Path(_local_db_path).exists(),
     _local_db_path or 'UMLS_LOCAL_DB_PATH not set')

# NLI model
print('Loading NLI model (downloads ~400 MB on first run)...')
_nli_ok = _ensure_model()
_chk('NLI model loaded', _nli_ok, getattr(_hc, '_MODEL_NAME', 'not loaded'))
if _nli_ok:
    print(f'  NLI model: {_hc._MODEL_NAME}  device={getattr(_hc, "_DEVICE", "?")}\n')

# Report
_W = 52
print('-' * (_W + 22))
for _status, _label, _note in checks:
    _icon = {'PASS': 'v', 'WARN': '!', 'FAIL': 'X'}[_status]
    print(f'  [{_status}] {_icon}  {_label:<{_W}}' + (f'  ({_note})' if _note else ''))
print('-' * (_W + 22))
_n_fail = sum(1 for s, *_ in checks if s == FAIL)
if _n_fail == 0:
    print('All checks passed.')
else:
    print(f'{_n_fail} check(s) FAILED — fix before running stages 1-5.')

In [ ]:
# ── 4. Stage 1: LLM CoT generation (5 chains) ────────────────────────────────
MODEL_OPTIONS = {
    'Claude Haiku (fast, recommended)': 'anthropic/claude-haiku-4-5',
    'GPT-4o Mini':                      'openai/gpt-4o-mini',
    'Gemini Flash':                     'google/gemini-flash-1.5',
    'Llama 3.3 70B (free)':             'meta-llama/llama-3.3-70b-instruct:free',
}

_dd = widgets.Dropdown(
    options=list(MODEL_OPTIONS.keys()),
    value='Claude Haiku (fast, recommended)',
    description='Model:',
    layout=widgets.Layout(width='400px'))
display(_dd)

BATCH_LLM_QUESTIONS = [
    'Does aspirin reduce the risk of myocardial infarction in patients with cardiovascular disease?',
    'How do ACE inhibitors reduce blood pressure in hypertension?',
    'What is the role of TNF-alpha in rheumatoid arthritis joint destruction?',
    'How does chronic kidney disease progress to end-stage renal disease?',
    'What is the mechanism of action of selective serotonin reuptake inhibitors?',
]

MODEL_ID = MODEL_OPTIONS[_dd.value]
PREFER   = 'openrouter'
LLM_CHAINS = []

print(f'Generating {len(BATCH_LLM_QUESTIONS)} LLM chains with {MODEL_ID}...')
print('=' * 70)

for qi, q in enumerate(BATCH_LLM_QUESTIONS):
    print(f'\n[{qi+1}/{len(BATCH_LLM_QUESTIONS)}] {q}')
    t0 = time.time()
    try:
        cot     = generate_cot(q, prefer=PREFER, model=MODEL_ID)
        elapsed = round(time.time() - t0, 2)
        steps   = cot['steps']
        for si, step in enumerate(steps, 1):
            print(f'  Step {si:2d}: {step}')
        print(f'  -> provider={cot["provider"]}  model={cot["model"]}  steps={len(steps)}  {elapsed}s')
        if cot['provider'] == 'local':
            print('  WARNING: provider=local — API call failed. Check your OpenRouter key.')
        LLM_CHAINS.append({
            'id':       f'llm_{qi+1}',
            'label':    f'LLM {qi+1} - {q[:55]}',
            'source':   'llm',
            'question': q,
            'expect':   'unknown',
            'steps':    steps,
            'provider': cot['provider'],
            'model':    cot['model'],
        })
    except Exception as e:
        print(f'  ERROR: {e}')
    time.sleep(0.3)

print(f'\n{"=" * 70}')
print(f'Generated {len(LLM_CHAINS)}/{len(BATCH_LLM_QUESTIONS)} LLM chains.')

# Validate chain 1
if LLM_CHAINS:
    s = LLM_CHAINS[0]['steps']
    checks_1 = {
        'At least 3 steps':          len(s) >= 3,
        'Steps non-empty':           all(isinstance(x, str) and x.strip() for x in s),
        'Steps > 15 chars':          all(len(x) > 15 for x in s),
        'Real LLM (not fallback)':   LLM_CHAINS[0]['provider'] != 'local',
    }
    print()
    for name, ok in checks_1.items():
        print(f'  [{"PASS" if ok else "FAIL"}]  {name}')

In [ ]:
# ── 5. Prewritten test cases + merge into ALL_CHAINS ─────────────────────────
#
# 5 hand-crafted CoTs covering specific leakage patterns:
# CONTROL            -> all signals LOW  (negative control)
# GRADUAL_LEAKAGE    -> UMLS density + specificity drop; NLI stays high
# CONTRADICTION      -> NLI catches abrupt wrong claim
# VAGUENESS_ESCALATION -> UMLS signals drop sharply; NLI misses it
# COMPOUNDING_OVERGEN  -> all three signals fire

PREWRITTEN_COTS = [
    {
        'id':       'control_metformin',
        'label':    'CONTROL - Correct metformin mechanism',
        'source':   'prewritten',
        'question': 'What is the mechanism by which metformin lowers blood glucose?',
        'expect':   'low_risk',
        'steps': [
            'Metformin is a biguanide drug that primarily acts on the liver.',
            'It inhibits mitochondrial complex I, which raises the AMP:ATP ratio in hepatocytes.',
            'The elevated AMP:ATP ratio activates AMP-activated protein kinase (AMPK).',
            'AMPK activation suppresses SREBP-1c and downregulates gluconeogenic enzymes PEPCK and G6Pase.',
            'Reduced gluconeogenesis lowers hepatic glucose output, directly decreasing fasting blood glucose.',
            'Metformin also enhances GLUT4-mediated glucose uptake in skeletal muscle, improving peripheral insulin sensitivity.',
            'Together, these mechanisms lower HbA1c without stimulating insulin release or causing hypoglycemia.',
        ],
    },
    {
        'id':       'gradual_leakage_aspirin',
        'label':    'GRADUAL LEAKAGE - Aspirin overgeneralization',
        'source':   'prewritten',
        'question': 'How does aspirin prevent myocardial infarction?',
        'expect':   'umls_drop',
        'steps': [
            'Aspirin irreversibly acetylates and inhibits cyclooxygenase-1 (COX-1) in platelets.',
            'COX-1 inhibition blocks the conversion of arachidonic acid to thromboxane A2.',
            'Reduced thromboxane A2 impairs platelet activation and aggregation at sites of vascular injury.',
            'Decreased platelet aggregation reduces the likelihood of occlusive arterial thrombus formation.',
            'This antiplatelet effect broadly reduces the tendency for blood to clot in vessels.',
            'Therefore aspirin provides general protection against clotting-related cardiovascular conditions.',
            'Aspirin can consequently be considered a broad protective agent against most vascular diseases.',
        ],
    },
    {
        'id':       'contradiction_statins',
        'label':    'CONTRADICTION - Statin mechanism with abrupt wrong claim',
        'source':   'prewritten',
        'question': 'Do statins reduce LDL cholesterol?',
        'expect':   'nli_contradiction',
        'steps': [
            'Statins competitively inhibit HMG-CoA reductase, the rate-limiting enzyme in the mevalonate pathway.',
            'Inhibiting HMG-CoA reductase reduces endogenous cholesterol synthesis in hepatocytes.',
            'Lower intracellular cholesterol triggers upregulation of LDL receptors on hepatocyte surfaces.',
            'Increased LDL receptor density enhances clearance of LDL particles from the bloodstream.',
            'However, statins do not significantly lower LDL cholesterol levels in clinical practice.',
            'The modest reduction in hepatic synthesis is quickly compensated by increased intestinal cholesterol absorption.',
        ],
    },
    {
        'id':       'vagueness_escalation_insulin',
        'label':    'VAGUENESS ESCALATION - Insulin resistance molecular to vague',
        'source':   'prewritten',
        'question': 'How does insulin resistance develop in type 2 diabetes?',
        'expect':   'umls_drop_sharp',
        'steps': [
            'Insulin binds to its receptor on skeletal muscle, triggering autophosphorylation of the receptor tyrosine kinase domain.',
            'The activated receptor phosphorylates IRS-1 at specific tyrosine residues, creating docking sites for downstream proteins.',
            'Phosphorylated IRS-1 recruits PI3K, which phosphorylates PIP2 to generate PIP3 at the plasma membrane.',
            'PIP3 recruits PDK1 and Akt, leading to Akt phosphorylation and downstream activation of GLUT4 vesicle translocation.',
            'In type 2 diabetes, this signaling cascade becomes progressively impaired at multiple steps.',
            'The impairment leads to a reduced cellular response to normal circulating insulin levels.',
            'This reduced response results in elevated blood glucose and broader metabolic dysfunction.',
        ],
    },
    {
        'id':       'compounding_overgen_af',
        'label':    'COMPOUNDING OVERGENERALIZATION - AF anticoagulation cascade',
        'source':   'prewritten',
        'question': 'Should patients with atrial fibrillation take anticoagulants?',
        'expect':   'all_signals',
        'steps': [
            'Atrial fibrillation causes disorganized atrial electrical activity and loss of effective atrial contraction.',
            'Blood stasis in the left atrial appendage during AF promotes local thrombus formation.',
            'Thrombi originating in the left atrial appendage can embolize to cerebral arteries, causing ischemic stroke.',
            'Direct oral anticoagulants such as apixaban and rivaroxaban reduce stroke risk in AF patients by approximately 65%.',
            'Since anticoagulation reduces thromboembolic events, all patients with irregular heart rhythms should receive it.',
            'Any patient diagnosed with a cardiac arrhythmia is at elevated thromboembolic risk and warrants anticoagulation therapy.',
            'Anticoagulation is therefore broadly indicated for patients with cardiac disease to prevent adverse vascular events.',
        ],
    },
]

ALL_CHAINS   = LLM_CHAINS + PREWRITTEN_COTS
BATCH_RESULTS = []

print(f'{len(LLM_CHAINS)} LLM + {len(PREWRITTEN_COTS)} prewritten = {len(ALL_CHAINS)} chains total\n')
for c in ALL_CHAINS:
    src = 'LLM      ' if c['source'] == 'llm' else 'Prewritten'
    print(f'  [{src}]  {c["id"]:42s}  expect={c["expect"]}')

In [ ]:
# ── 6. Stage 2: Concept extraction ───────────────────────────────────────────
def _strip_md(text):
    text = _re.sub(r'^#+\s*', '', text, flags=_re.MULTILINE)
    text = _re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = _re.sub(r'\*(.+?)\*',     r'\1', text)
    text = _re.sub(r'`(.+?)`',       r'\1', text)
    return text.strip()

print(f'Stage 2 — Concept extraction ({len(ALL_CHAINS)} chains)...')
print('=' * 70)
t0 = time.time()
for ci, chain in enumerate(ALL_CHAINS):
    chain['steps_clean'] = [_strip_md(s) for s in chain['steps']]
    chain['concepts']    = extract_concepts(chain['steps_clean'],
                                             scispacy_when='never', top_k=5)
    n_valid = sum(1 for step in chain['concepts'] for c in step if c.get('valid'))
    print(f'  [{ci+1:2d}/{len(ALL_CHAINS)}] {chain["id"]:42s}  {n_valid} valid concepts')
print(f'\nDone in {time.time()-t0:.1f}s')

In [ ]:
# ── 7. Stage 3: NLI entailment ────────────────────────────────────────────────
print(f'Stage 3 — NLI entailment ({len(ALL_CHAINS)} chains)...')
print('=' * 70)
t0 = time.time()
for ci, chain in enumerate(ALL_CHAINS):
    chain['pairs'] = build_entailment_records(
        chain['steps_clean'], chain['concepts'])
    n_contra = sum(1 for p in chain['pairs'] if p['final_label'] == 'contradiction')
    print(f'  [{ci+1:2d}/{len(ALL_CHAINS)}] {chain["id"]:42s}  '
          f'{len(chain["pairs"])} pairs  {n_contra} contradiction(s)')
print(f'\nDone in {time.time()-t0:.1f}s')

In [ ]:
# ── 8. Stage 5: UMLS signals ─────────────────────────────────────────────────
print(f'Stage 5 — UMLS signals ({len(ALL_CHAINS)} chains)...')
print('=' * 70)
t0 = time.time()
for ci, chain in enumerate(ALL_CHAINS):
    chain['density']     = score_density(chain['concepts'], chain['steps_clean'])
    chain['specificity'] = score_specificity(chain['concepts'])
    den_sl  = chain['density'].get('slope')
    spec_sl = chain['specificity'].get('depth_slope')
    ds = f'{den_sl:+.4f}'  if den_sl  is not None else '   n/a '
    ss = f'{spec_sl:+.4f}' if spec_sl is not None else '   n/a '
    print(f'  [{ci+1:2d}/{len(ALL_CHAINS)}] {chain["id"]:42s}  '
          f'den={ds}  spec={ss}')
print(f'\nDone in {time.time()-t0:.1f}s')

In [ ]:
# ── 9. Results summary ────────────────────────────────────────────────────────
rows = []
for chain in ALL_CHAINS:
    den  = chain.get('density',     {})
    spec = chain.get('specificity', {})
    rows.append({
        'ID':         chain['id'],
        'Source':     chain['source'],
        'Expect':     chain['expect'],
        'Den slope':  round(den.get('slope',  float('nan')), 4),
        'Den risk':   round(den.get('overall_risk', float('nan')), 3),
        'Spec slope': round(spec.get('depth_slope', float('nan')), 4),
        'Spec risk':  round(spec.get('overall_specificity_score', float('nan')), 3),
        'Leaps':      len(spec.get('abstraction_leaps', [])),
        'Contra':     sum(1 for p in chain.get('pairs', [])
                         if p['final_label'] == 'contradiction'),
    })

df = pd.DataFrame(rows)
df.style.background_gradient(subset=['Den risk', 'Spec risk'], cmap='RdYlGn_r')

In [ ]:
# ── 10. Stage 6a: NLI P(contradiction) matrix, all chains ────────────────────
ncols = 2
nrows = math.ceil(len(ALL_CHAINS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5),
                         squeeze=False, constrained_layout=True)

for ci, chain in enumerate(ALL_CHAINS):
    r, c    = divmod(ci, ncols)
    ax      = axes[r, c]
    steps_c = chain.get('steps_clean', chain['steps'])
    pairs   = chain.get('pairs', [])
    n_steps = len(steps_c)
    mat = np.full((n_steps, n_steps), np.nan)
    for p in pairs:
        pi, pj = p['step_pair']
        if pi < n_steps and pj < n_steps:
            mat[pi, pj] = p['probs'].get('contradiction', 0)
    ax.imshow(mat, vmin=0, vmax=1, cmap='RdYlGn_r', aspect='auto')
    ax.set_xticks(range(n_steps))
    ax.set_yticks(range(n_steps))
    ax.set_xticklabels([f'S{i}' for i in range(n_steps)], rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels([f'S{i}' for i in range(n_steps)], fontsize=7)
    ax.set_xlabel('j (hyp)', fontsize=7)
    ax.set_ylabel('i (prem)', fontsize=7)
    for p in pairs:
        pi, pj = p['step_pair']
        if pi < n_steps and pj < n_steps:
            val = mat[pi, pj]
            ax.text(pj, pi, f'{p["final_label"][0].upper()}\n{val:.2f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if val > 0.6 else 'black')
    ax.set_title(chain['label'][:38], fontsize=8, pad=3)

if len(ALL_CHAINS) % 2:
    axes[nrows-1, 1].set_visible(False)
fig.suptitle('6a - NLI P(contradiction) per Step-Pair  (E=entailment  N=neutral  C=contradiction)',
             fontsize=10, y=1.01)
plt.show()

In [ ]:
# ── 11. Stage 6b: UMLS concept density per step, all chains ──────────────────
ncols = 2
nrows = math.ceil(len(ALL_CHAINS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, nrows * 3),
                         squeeze=False, constrained_layout=True)

for ci, chain in enumerate(ALL_CHAINS):
    r, c    = divmod(ci, ncols)
    ax      = axes[r, c]
    density = chain.get('density', {})
    if not density.get('configured'):
        ax.text(0.5, 0.5, 'UMLS not configured', transform=ax.transAxes,
                ha='center', va='center', fontsize=9, color='grey', style='italic')
        ax.set_title(chain['label'][:38], fontsize=8, pad=3)
        continue
    per_step = density.get('per_step', [])
    xs = [row['step_index'] for row in per_step]
    ys = [row['density']    for row in per_step]
    ax.bar(xs, ys, color='#4C72B0', alpha=0.6, width=0.6, label='Density')
    ax.plot(xs, ys, 'o', color='#4C72B0', markersize=4)
    if len(xs) >= 2:
        coeffs = np.polyfit(xs, ys, 1)
        ax.plot(xs, np.polyval(coeffs, xs), '--', color='#C44E52', linewidth=1.5, label='Trend')
    slope = density.get('slope')
    if slope is not None:
        ax.text(0.98, 0.97, f'slope={slope:.4f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=7,
                color='#C44E52' if slope < 0 else '#2ca02c')
    onset = density.get('leakage_onset_step')
    if onset is not None:
        ax.axvline(onset, color='red', linestyle=':', linewidth=1.2, alpha=0.7,
                   label=f'Onset S{onset}')
    ax.set_xticks(xs)
    ax.set_xticklabels([f'S{x}' for x in xs], fontsize=7)
    ax.set_ylabel('Concepts / word', fontsize=7)
    ax.set_title(chain['label'][:38], fontsize=8, pad=3)
    ax.legend(fontsize=6, loc='upper left', framealpha=0.6)
    ax.grid(axis='y', alpha=0.3)

if len(ALL_CHAINS) % 2:
    axes[nrows-1, 1].set_visible(False)
fig.suptitle('6b - UMLS Concept Density per Step  (negative slope = grounding loss)',
             fontsize=10, y=1.01)
plt.show()

In [ ]:
# ── 12. Stage 6c: UMLS ontology depth per step, all chains ───────────────────
ncols = 2
nrows = math.ceil(len(ALL_CHAINS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, nrows * 3),
                         squeeze=False, constrained_layout=True)

for ci, chain in enumerate(ALL_CHAINS):
    r, c        = divmod(ci, ncols)
    ax          = axes[r, c]
    specificity = chain.get('specificity', {})
    if not specificity.get('configured'):
        ax.text(0.5, 0.5, 'UMLS not configured', transform=ax.transAxes,
                ha='center', va='center', fontsize=9, color='grey', style='italic')
        ax.set_title(chain['label'][:38], fontsize=8, pad=3)
        continue
    per_step = specificity.get('per_step', [])
    xs_all   = [row['step_index'] for row in per_step]
    ys_all   = [row['avg_depth']  for row in per_step]
    valid    = [(x, y) for x, y in zip(xs_all, ys_all) if y is not None]
    if valid:
        vxs, vys = zip(*valid)
        ax.plot(vxs, vys, '^-', color='#55A868', linewidth=1.8, markersize=5, label='Depth')
        if len(vxs) >= 2:
            coeffs = np.polyfit(vxs, vys, 1)
            ax.plot(vxs, np.polyval(coeffs, vxs), '--', color='#C44E52',
                    linewidth=1.5, label='Trend')
    slope = specificity.get('depth_slope')
    if slope is not None:
        ax.text(0.98, 0.97, f'slope={slope:.4f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=7,
                color='#C44E52' if slope < 0 else '#2ca02c')
    for leap in specificity.get('abstraction_leaps', []):
        ax.axvspan(leap['from_step'] - 0.3, leap['to_step'] + 0.3,
                   alpha=0.15, color='orange',
                   label=f'Leap {leap["from_step"]}->{leap["to_step"]}')
    if xs_all:
        ax.set_xticks(xs_all)
        ax.set_xticklabels([f'S{x}' for x in xs_all], fontsize=7)
    ax.set_ylabel('Ontology depth (ancestors)', fontsize=7)
    ax.set_title(chain['label'][:38], fontsize=8, pad=3)
    ax.legend(fontsize=6, loc='upper left', framealpha=0.6)
    ax.grid(axis='y', alpha=0.3)

if len(ALL_CHAINS) % 2:
    axes[nrows-1, 1].set_visible(False)
fig.suptitle('6c - UMLS Ontology Depth per Step  (negative slope = abstracting, orange = leap)',
             fontsize=10, y=1.01)
plt.show()